# ZUNA follow-up test (probe 2)

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save. Then **Runtime → Run all**.

- Allow Google Drive when asked: progress is saved there, so if Colab disconnects just **Run all** again and finished parts are skipped.
- Expected time: about **1 hour per part, 2 parts** (~2 h total). It prints "still running" every few minutes.
- At the end a file **`probe2_results.zip`** downloads. Attach it in the chat.

In [ ]:
# 1. Check the GPU and install ZUNA (2-3 min)
import subprocess, sys
gpu = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True, capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "NONE - set Runtime > Change runtime type > T4 GPU, then Run all again")
assert gpu, "No GPU"
subprocess.run(f"{sys.executable} -m pip install -q zuna mne", shell=True, check=True)
print("ZUNA installed")

In [ ]:
# 2. Save progress to Google Drive (allow access when asked)
from pathlib import Path
try:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE = Path("/content/drive/MyDrive/eegpp_probe2")
except Exception as e:
    print("Drive not available, saving locally only:", e)
    SAVE = Path("/content/eegpp_probe2")
SAVE.mkdir(parents=True, exist_ok=True)
print("Results folder:", SAVE)

In [ ]:
# 3. Unpack the experiment code (embedded, ~16 KB)
import base64, io, zipfile, sys, os
PAYLOAD = "UEsDBBQAAAAIAJeGUV0GNNx67QkAAMIYAAAaAAAAc3JjL3BoeXNwcm9iZS9ydW5fcHJvYmUucHmdWFtv2zgWfvevIFQsKnUUJZkC++CFgJktOhtg2mmRdtpgDUOQJcpirduQVC4N8t/3O4eyLF9adFZAYoo8PPerPM97r9uVFPK+k1rVsrFz8d8///hV3BpRqUamWmhZSC2bTBrRNnhLK3EuutRaqRuDpem1bteplTjLWp2rZm2i2ez1rdQPwrdapVUo6tRsQpG13UMgFCGSIipUIe7K1kihmqJdPF+luXm+FGVb5UbYMrV867kRpcpz2QhZyczqNpdA/5uqwE+qpeh0C9aMzIFFZGXfbEwoZJqV7kWYrlJWtOAGOKX4z/s/jfCJPos53BaQno6Cf810e2eiLwaipk0uqnY9vGiS704ryI23AtILySIymVCYVqTCQoVnbW/FRsrOiEI1ypRg7a7Vm2j29t2r3+NLoOmqlNTJHKwemMWGtACkXVulVoGgX7UZVG2lAfcF895Vfb2CeoNo5nnerNBtLZKk6G2vZZIIVXettmC7aS3jMLPZsEcibNet2a5M2VtVjW/9atDGuPMwLkmwca3B/SrNNo4DuEJZqdWW/Hu8jnSbvu4eRGpE081ms1wWQveNnxXruchVZkMBXc35ClykzTZzsWrbSsTit7QyMpjPBB7ARPUmV9rvYIXGmvij7iWMfK+MTdoNvwYMCnPhMqH2s7Yp1DoGLYc6pn/h4CHxYukumPRWJu5WldarPJ3DOWC/c+FtTe8FEZldJlbeW592ohxiGRgIqFUD17TxZSggXdpXNjZWBw65vQBa0lxE//xhUz84sehBBMBYTvRxk/kajRGRxgrvEcaI5L3MepuuKvkkzqB51YG+sWlVibO/xNe+SUXdSA/OWMqqGtSUlTKb6ojpOvMAeNxiW0ZiNH1DNukoOYwgAI+MtKSvpILzV773+vr63bW3Q1wVEJkvR6teVXlSaAkFyzQvlKxyfwf5BthbE3Lsk/aLhffGW4a8wMF2Scfecrz1CqBvIlOmnVxc7LabtEY8AQs8wqcrUVYmvLkjqBuyctNFGoHd1tFgrwT75JELz0iZe8vdhQ+XgGcNRAZRV8nEtL3OZJK1t/6bHZyFBbeAWKcVA3y4nGilRm4ixrVk1v1XoTibnNcNXSH+CfAX/ok+Hh8Pi3OSgqPQdzuB+ElcyrOfxQs6kQ8gEADq1WyHApmUFLTwPZNx7oasCIxAFK0WNedO0gHDecvFBGq5BHbcQ5xn5XfvOIjlziyfieLula4lobiim3xp3+c/EXTGUBmBwE5rlgRRknGcYPNquXfnc5R2nWxyn2P+U/wpFC9eOFO48pUUaa2qB5997QPCFBBXQTCxHhUos8/o24mXXe7zvxnFJpn3BUC1id1Zkzi00ArSgb8JliSEMhytKKX+AVTIOStAiUM1PDjbI/FMAB+SO6IpF84dBRUKTvdzcZdWG5QK3fbrktJ7UVSAQ9lVVL1dFUHJraRYa5XvYQaWBLhlWrsoQf4EWtngz18gRiKUyLp3hcV/63wg2ZlpQ763EefnUB6cMZjE0ag65Mqd9vBypMAtpNrhbTbBMczAL0f8yPZCgYXN3FdMn9as9C1V36PWBf7Lv8lq7Q3q/qOdZMHpczNGdZ1upLOHXxWhQBkhrKEY0gZUlbMRoSbnZMRcTP+CKDX2AREPjRZVm9qXPwcniTlr73kz5a+48DaPm6fkkQjiR80vXuZPkELFClzEAyux4+cmvglF2cWO61Ktyy41xr/ZMsEcBFPvrzvrfDZaS+u7UE5gaSct6FSy8Xk3CEYX5O6KAQRuGUbCMNDIQwaPG9H3jbIu7aDWu17QLCxShfKWsBBd/El8CcQ/9uiQC9idC9CRuzqcfdmdAQNii+MC2hrSHMG43ozUMi0A0RATPjMWiB95nola3QuyseIoE3fKlmBg0v2NJKjPg7jU0Pgt6nVzqzT6Bdbt++t3/36dfH53/Ts54bmtu3O2U0KXvGkr4PySEsfQijDE1GxNsu56M7Wc2wHmSVkhO7omGKUjvfcvB48tqHlmK+NOMkKxSQbMiGMGdaljWhWpP97PlmgHKF+xSoEjZls6Be/DoHfouwS1jaH8SWckztAsUcn658V4ZduaTXoGLg2T5HARTmiFO3kPUoa9PNGHbZ+tghjJIlPzjBLIiGo//RZcx9nKMAsGmATC7Ddu3FNHukZOQsCpBqG6blo06BJjkjYHjdiA9ER/ewhTyQO1bxVCoVWrYBx/jtMlGiJcdW0RJjD/OANdsYcgdha1Wk5L3gTFwqMBjU234NZqUS6ZgfJEXabnhpU1FGOZbrBqSWFdb6ep4MZb7ucnKtAny8KkK9h3ztMVgoQmg50LtM87gs+J+efLp6R+rNVTkj1mT4lO72gc9U7iobZXtdF1ever1umDf8NGbVAIkAFWmF/jbRsckc/6Jw5O80c23Wb8Yh8G2eB4KKAHMzb5wePTSQ0VpCHGe1olkJPbSBYIdTAnyRNITkx3WkL/+TA2/JAIpLfd2CoT8hAfTS6GBFr+LVQQbFFEZBzODoQamS3JU5v68Al6P27q6aFseaKP4HmGZqLEYqY028kG41SCOMPMJKujS2t0Th3HGStxsZ7PXUJ0nr6e9CW8HRx7/WCgKRnfoQ3HxEFc0ZSGgSOejB5Iv5cXF8g+GWq7sbIzwyktqb9wROOB9v+RCaxGGa5ds3AFpX6mZm8MjXCXA3BEP0cIhk8xP5oCrO5tOSaBsSFxrxrjRZHcoivGQIuMcDIVfK4X3ieEONXqZ2IAPsNFgRkuVc0RSXhxfiJTbnX0N3MIeR+N0GRRdlDKJN9NIcdkR67G7g5IF4usjOjjwb1fBjsdOv0ul641eeY+EaELQM4MRVpRyD4I0uMZKVLmR8Q+Unyrxl+w6scZhme4bnxlgvTxwDG2PA5Je8WeARyLq1DMPx6L9XUYFGqZgly3mDMY3NvfriNjMfG72fTlRXBINBQXx3TRXmy/4LBfxuQXpG7y/w2/bXhJXS+/YeENrhvXyi0SPnaOftIew8P6SCgch57ZbaBU+1/rEEoIwglM4j4DHIGyOIuL5SA23ztdwe4aN4q4L37UB6r8jn5W0tgzN7HSq1udmbptbel9wzehKvijY+Pxrnni0nzIG+LnrlmKXwZbDuG04/OUAUZHxXofYNuQjh9Jv/lxjCAOlMA9oPv+RnE+mXQwFGsbU9bjLiymps41cqGrZrzDq4BSXNNb7B02kJeugTygeqKNpKfTqFt+4fgRj2j60PLuGmb47NP545l/tusv9wCCp7l43DH1NDRoMO6pHgJZ4xvMzqPL4okkgtWLqjflid5vv6U8qCHfay5Z4e2G/YJO+EDeZxJj12v+wTwzPwnO31/3T5gEH44fgCP4dJ3aBDgn6nWqnd45lo1P3ceyH5kHDoyoMUjohpDM/gdQSwMEFAAAAAgAslRRXQAAAAACAAAAAAAAABkAAABzcmMvcGh5c3Byb2JlL19faW5pdF9fLnB5AwBQSwMEFAAAAAgAeVVRXX3QSvG3BQAAEQ0AABYAAABzcmMvcGh5c3Byb2JlL3N5bnRoLnB5vVZbb9s2GH33r/igh4FKZEVOmm4IoGJtl2wDnAxI8hDAMARGomw2EqmRdJz01+9QlC9p2gLDihnwjfqu53wXRlF086zcUjhZ0vn571Qb3RIn2y2FkSVvaCl4Ra2uREPs8uqc3owb/ixMkBCJV7q8/PO3D/T2zbhccqUgWPIuTqMoGvXWiqJeuZURRUGy7bRxBCntuJNa2dFoOFOrtnsmbkl1o6CXNnBdS9FURXC2Ud+4LD7+8f7q6nx6Mxpd3FBOx6dv02w0GlWipvuVhF5thCi2ZtjCyKpo2zOqG80dNCZZmsVnI8IL4V5AeqyNFCoER16TelViHxO6TOgkTog/CsMXgoyoEZQqAULXrCyJRpTOAKkjq1emFNRpK/skeyy8kyGBVon+L75TK1zR6AWifBQNi86vr/+6juIgrWqNIL1UaQR3ovAnrJHWsVcYIDCLdP/OL24SKpeFe+6EzSMhFnvWem+tRn4LwXq7gTGbtvxBFNZxVXFTbUWi7ckky06jOFga2AiRBcX+pOjLhJksj/jK6Sjpi6cwvJIruz3zceT+YzBmysESYlt1xaNuVi0s9hDCMC8FA5D5wF0yeM839ddKVQGQ/DTNEhJPZbOqRH7see3N1+tqP9Bam7VPx8KLp4b5QBJyhiubX2kFgwgoxzuhe9HuvIhFfsEbi59ANL81KxHsT2EdPmYRLEbzWVRxx6N5yq3Hn/V1FgSN2UqaEpIZhI3Bjy/OUFxO6Wg+p/GQKuSyaB6MoAJgZnPed6LdGOPVIPUgRAcpZszsLKHjOb2jMQCZxPQTMdWljVS8WaRKmxYyKOgnaXM8fUdZenJKB72boWiqJ1iCztp7ZN5yDG/b1KepEXbJkWoj1FeLcjzxXeMDga09RaQ3TVvBFQNv3m4lW7sHLDgPnrkx/JnNPFGzqFz6dCXSbbRH7OxkTuCUJMqKQOIiBLInHMfz+KtQotqMokqWjk3zaeId5ngnYCoHcj7aF8Ua5kon1QNTZ3jgErodvo1aJFSjEneTJUtP4fadT8A3EFLYjhkFpUp0Ah/K0eSoJidbQRYDV1jSNcaOWrgl3Sa0UtLRIzeSY8xsp0gdgKlrlxp8+K5nEJ7QEV3cBPR42+3zVoNbH18QwrH92zhfCi24b1ctq0P8MQjbts7dzo30fpg3eoCyUqF2eMOs/CxyBrMe9DqOYzqkySdfQd+RgQ+V38b7LNwhqrvUuopNXhVDwN0sn92y/QbyWc5+QWon8bcg/4ApNm5kK52osBqMdZaYSBcp8aZb8viMFBT0enwPQVJaWtGTYv0qbPSahMJ01p34D5SUYLufmh4bWKl9+x3UqP4tQpP4C/bEU8fQur4nGatnfkD5uRBMxTA+AVl0cEDH/x9lgCLUNxR9e95bFpoi8RW7bYU8S4/jXVR38ADNf0d6rXEDGVaBZQ2K9OE1/7v9fQPpDmguMN5sv7Fv43CheUCfd7oBn2uJzsKgqHCKEdLv+aP9ne8LIBSbLA+3zVkiBit2i9yPsKaeRdNhmlz2s7CfhLPJfJ+Lz8Joy9jmYTb3UQVc4N8O9VAutcSiww3jAUmJrsHaCwsniLqyx/xnz1PohIcN3J5AzG086WnYnYfCxHD8hBLz8/Gz7Jj3iX1XDqj5lx5i2Cf/JN57fNQnsr819O7xHR32j/XKCcOmftB/mtOvpL3TUFW43gy2Mz9eTuKXvZ/0QGz6XJTaVFItXtBtcaoqO4zXocpMa4vV427k+qUPUWWK6v7l6ZDsLf7CHBusIbpNbw4x+A39uuReAopCzgfXMDAR47fbMt4IHL7G8y7Qj17dyHrlzLcuG4eQYeY4+w4yFvfOopY1uzvbG3HYW9wth0Xl58OQ6xc3TVmH26S05IV25P/YO+aPvWcavh5Ckzq95uv3/U3gLiSbEG5K99qK/MWdGTqpR4oFWDSE1gZjv58r39IJcHuN0T9QSwMEFAAAAAgAslRRXYG3sDDjBwAAMhEAACEAAABzcmMvcGh5c3Byb2JlL2xlYWRmaWVsZF9zcGhlcmUucHmNV1tv47gVfvevOFAfKmVsj51b00y16Gw2aQdIYmMmBQoYhkBLtEVEErUkldvD/vZ+vMh2kp2gBmRJh+S5n+8cRVH047kxJTcip0o0nCl6kFVX81Eum6LLjZAN1bLg1Tn9l1K6phf6RI0Umo8Hg7uSU8VZQWvBq4JyWXNNayVrurm9/Kum41HFnrki3ZZciZxVVNrdjh/FK8VEQ5/p4scV/vV9V1X0eaCxrU2IP7CqY4YXxAxBQTo9Jl7x3CgcplZqYVXTJNdudV4+ayFvuaHLy3/d3Hz77VfKWTukR2HKgWKFYFX1TFIJ3liehWhlBV1hHCNdckieUl7Tilfy0TEUTWMVd0rpTq1Zzsd0K0nxXKqCF4OCGUZCYyMc9sCLLzhmX7U7bviTWUl5T2upHpnqbXZLJZfqmTT3zmVaw916PIiiaOB8l2XrznSKZxmJupXKEGsaaZgzeDAItKar22ecpqYdDHqjs4t/f729vbz+gVDFA8Ivuro4oauLI1xTXC+4DnEd4zolLGEFC6CDDKolzkGdgzwHfY6FOVbmWJqfUhR4tuDVgld7SF+v/obrCNcLrmNcZwTSFaRCKPZhGyRC4CnZpTus3Z0Rbvb/73Q3nfRs7+Ygzs8IN+gAFaABFIB8iId0uzTD2gwrM9BnoM7OaDYlvM0O6dtLNEjGuq2EiZPBYFDwNTJzJRuW5yIreS1cIvK4OUfYzJBeslrgeV1JJFlKo8l4mtDoF7h03BRMKfZ87jWLoj/+0wjEsqauEYYeEDyptEsvVMQvqWdFcde2SBub5cnYBtSebo0G88XSvdR4hOy4oQM6RNrHUxr5w0mCyjpzmzayKniDnVCkFdgZH2EXXvTvysQnSeJ2QR0SYEaKNRse14lX1v5ecNYyPrRnBfhOxicJpNXbHWKNTf8IHthS7Q+Fb0TT8S1ReUWcbKctmL4k22WYN2YwuyniWGEJW3Op42DDAYlkSIGuRfOa/hJMURwJ39gtzusxePYBXHWiKjKLMg5k4ibTslM51y6GUO10MhkiIGolNT8nFF0F4hWrNE+20fvu+RciN/H1kPIyaxiKzj0BS4akVb57KIQagvlaJj7ADFUKAWzDoegaCdTkFv0s72sqUYK6ZC2n+GJIN4k9aTEUQbepEH+lOvlCyBkgg+IW3YA/Rvo8qjl45xZwYbzNLi0dW28iBBcea+mBKcEgVgPWGsAUaePAEbIspmhYQ0ythFEM2GJ5A1GC8e4eUKNuuFccD2PNTVbJDZz7wKs4uvz+ffY9spkBtOk9CtDFX/Tt9moW+VjVyA/ritTxyEuAE/aMa3bPM2hlC6fIwqY42lKmk8lJ4GBd2x9XHHZklhJXQpv4HZLBoXqt+O/p9HQynriImeeW6zTifLPH0FnTiw33xNvqqz5I9Ho6SuZAOVaTNGKdkdHQFW5m20WntzTLPHXZ4FN1Akb+/CJSk8iXtcp8N9stuc6no+VissQ+VkTLXYn9xbeXkW8vXh6hgB0rZJ/Fiz8Hrj75/daQtdgNrT5R3GsxQr1PppMEVea49VL3etxef+tZBQfBj12b+SkgiIO/0P9iiEpdCSmVbgumaVIrw4bJ6Zj62/AVpnz0C4mW+pL1yPZY7Ecr9FAoU3W2CcY2GkNCrjc6vZUNd2VrVRrCurrXANW18Vz/P2WQT+md6vjwY5WAj1g377XClqEFZfhrLZ548XNm1za+j8UiwllkSGRniWg5ZtpmdgwUdB3p9DgJceuRxbFFmN08k4Rxxs0Fju09b82Ws8pD7lllGxny9NrNcCO6HlvoidmT0CmK6p7zthC1dione3n6Dvf2uHy2eI2hkVWbsQUvi6yeYeK5e2M9xPoe4tF9YeO3iPLSloeAipW0yp4fLd/2tIr7YIfNSbJEf0ay7zcND+rpHqynP0USr0saUP9dVoSk7pN7YR26dAK3jcElu194BQ2+WUHprmIKGy3uoWTY+d4wgezcf/3TUYPBUey9a1lw7Wj6Nlju1AqnVu9PrT4+td91c9uzbeuuRBsz+ietxndDsgenSW9dAINc9t0o3nbLvo/uOusBrN2A+MhVOrXAjbdHUZgyK1Bn0xOQflKVmP9L1p90mO8pzKRxNJsCkqPZi/s/tP8YBaPkY147uYeOn2H63rKLLo6i8Lbbcjae7AaHX1l+v1GyQyP+hJ6rZLMhCVBuhcHXjCqfTVkPXZ92bbhFMYKOm+HwLD5OgE5OQHBeGBv8PALMhy+z1QYcMrtpO8F5Mp1TfGMrn3YutzzRrhUEC+hiSuYmiwO07AP/ebET9tYncdAPX0K6lhKTzWpnXluhB7HeRgQTFghUI47kZT/teTW9XokbM0Z+bHlv+BtdSEJp2KHwZfRlu/hWwwrDAIY6Q3/QbTyxuUf2g035j6rQ++xHjxtGXeBcnthOF3SDF15PPha339XlNnHDgzfw0aMUsuAQGBvvZ2zAMh+XdJvafq7lT22Mj4cTO20XKMNH6IPxPowftnScF+NQGjumu8k7h51uRgE7B54LX1KLvqLGoin4U9wkHiQbC5K2cg1GY22yaeG5J8stCG95Fx+7wIte2Ea6TBbnQ5osd8O91+iVgYUHmt5LO2t6q/vDAV/a98DUBtegqG2BQoh3UF/ow7eF+8b94eHTPlb4UMgO6RIHvsNegM0P20u2r5/JIss+CvaV6BUJCPEWHJLB/wBQSwMEFAAAAAgAl4ZRXce4RK+uCQAAHhgAABYAAABzcmMvcGh5c3Byb2JlL3Byb2JlLnB5rVhtT+NIEv6eX1HKfBgbnJAwmr0Tq5yGQyysxLwIOIa9iM127HbSF79dtwPDnO6/31PdbcchmZ3TaiMEwd1dXfVU1VNV7vf711Jk9GgoLh+FVqKI5SAXdbyUCZm11uVC1JK0jEudqGJhIhJFQvVSUqYKKTSWUqkljpEqUpVllMt6WSZm2OtZ0TWEZmQ/J7SiRFVlJg0FGnLKnLIyFrUqi6NSK1nU9ntIT6pekl4+18tcxXRI46OUapVLaLnWRpqoRzufeqnL9WJpdTPVEkrRUoqE8jKRWUQmFhlsqkv6Tedmtn78LaIqWxt6WipYaGRhSk1FqYwc9j6Jupa6MI32rLnB1TE0t7rZSwQU2tKK5uuavGEmL0ts5Gsrqhp5iRZPBaW6zPdaIAlqyHyeyY4/KLA3VVIPnDq4YUXCAHqR/QiVa48qpUpmiQmHvZvWc+7ECV2ItTEQBy8BHQGprR0HzZ0HvFLz/s7lZWo38V1k1KIQ2T7wHRYyr0qNfZUugRTwsI8ZAoSODRyHL50Wz1D2C/zho8hfvEdypWWi4hq+WcLkFnf5pZJxDQESlmoqC5oD7mGv3+/3GF6azdJ1vdZyNiPFatW4H1DZADO9nn9WrPPqmbEsqp47N2y2m+eiXvZ6vQ+zv1/QhI5Ho13l9n9e0VzEqwXCERY3AQ/cXzOIs/nide/0/afZzc8X708heDR883/JzMrFoCh1zn6woUgirzJVrxOOd4hOeuenxxCYZqWog6IaAqPgmA5oc9vBAR2H4ZbY86n49fgBdiYyhYOLRSZnTv4MURBchSfWKcD1fJpR9uvtA6UWcNlEnQC0tC4UHufZcxP/C62SNrt9QruVTqYP2V0s/iyi9xHNoP3V0CxFJe1TLeHBgtgUVZh1HvTjXEVJrgZ/i5N+RFf4CemIgjew8n3ojeCUnPmUDFYR3Uaki4W3o44t5H/BAevgoWOZzj7QDbukWa9UseqsdvWCrCP8Gpo6CcYRraSsEpWbya1ey0aZXKzkzCZhkKXYgx9VJMgMMGqRGCs0ohscd6w0OR4NR1gu9CyZ+3+Aopl8AOIbX3wsOqSMfMo80RrOV0EL9SgL5qylAA1+/HBOi6ycI3IWAnEYFGWXTaRP0NAltiltloFOpbTKGmTeo+yk3h6KWh24ODq4GdP8GSuFqfU63vLxFZDP0mn/qv/wOz6/xf+qqAMPUOuHn24c9k8R3WELYgI6i2yB0Fgsg5uxWz0dY+0Oh7Bu/q1tGsSZqgIcA5IWRLdTpdY4mkyo3xa6/knLQPfuEitkBYHIrZBzaUzvtiPsbDs2ZNaV7DO+I/cV3JIiLbJNKQT92I19jr0NdZyQpR6fNc1um0nfroevOoVulyyr8klq8jzqtKgEuC7IS80VfeWJ/vz8ImxFcvgBDFg4hGPkQmoTjKzvViHDaNeVseDCfCMZN2GE1uI54MWNqNLLcTwWGPVVTji53oThj1g86vqV9wQl+o0vykx206sReeH81BLESkUr5ocV88P0xGXPQ0Tl5sh9ijPBBbzJuoBXvTowqqVKpCArtwJbvvsWo7QC59l+fNh/HePn+6znPQ0A830IzL8Pwfx7GMwzRmC+OVLxkQ7BOS1uPR2luSom4+GoAxnvxz3v+OTBy8QADW6byskDlA9xzieFkZsUeJkgnez4k+PtOzH3Yt8fCb8/GILbLNG0hy+AYLmWb3bUP2MsutK68PJHC/RZdCeytTznBingu7b982clgC0pE1++IHIsBz8gIF6GiA2aWqPzZLLGhjN33HaEM5PsiMCfETcswcAVQxw5Hm3VX3vzAYw53Eg52AvWbdg2B2gBM9vZcM3tlNlNbT3rdN7rionUUmpoSysaV+s2bou5fEa+KYYOWyNE29hw0ZNFW6C86iiTR9uAHO6aOx6FDjb5LIMzi1nbVZiVCaoS/UMxs//wl6VKEln4HLYOdzaVmEomNHVVl5u3GXejqCsLGfjj4f7shOEcmTJ5EZmXPqfiZamgPxsIZcItJWSVwbbJT2hK5O+Favwy1xtp4c6VnOd6AZBtXe8mKvbTAOXNTOOHJmfDcHrS6POwYeF1PRRVJYskYEEyCbjbWIYWmCUDcxluuQoHmu644mll9tkBfxdhpzPFDg55IYEHhgyYaI3RKNiu+/UTxcw2g1tLeIQw+tK979u7+N7pHezjv5cPTUi7EQpqIaqdWkBfJQs54TzyKvKkgfUpcFNfZgF23YUPCDq7sZuc2MSRxk6wX9sA5Cd329B0BF46ge86DKqKx+C0UXKJLq0SxgT3KC/xZDR86xXz2AC9rmQGM1UZMPB/ZomoBR9uukEW4xq6iB6lnpdGTvrn19cfr/t85ysavPjAJLEaYDRF6uJ3XcZlhtEaTe6/yrk54tVw51Dv5pxZeHz8V57+XtE///Hh9LWht2SQ7osccwyPP8dvf6DLr95SjXxJG4+dMSwts1zLweZ1iW/Q0FprsWjn7EdlFPfWMkPHpssEHd9dSya5y4GvUpfoO50r8mmmTB0w+NBzOPK+u9ty1IZDBvwdIS01pxDQY0ER5eGW9l4LxrtR30u6h4T79sphLkXBZWPv7MOQWrwRzdW6ZmmXGzDu3B2DBoAWmYhczpIunwyxsRK9sEee5xafZcSXH2H4sm8WVG2sZC+uA3SZpkbWjC4iFl3CMyiaX1ZxjbDpKebMkIy+v3gD/haP37sxZTpybMJjyDS2vBFvCPXM9iixfS3D71pkHVyG7sAvzHS7+Lo1h+mldSPKURfzXxr+4dtnUuvgE5e0Fsn3wIFK2E0N/1hEOkFqx8JgDN99AlUY2KxDTI3N+4NBilmvNXWnXJjRxjz4+tbDMH6APM4PlK6Iv3SKSAUJn7j1MaMTHD/kZd6+88xF0Lf7K1bldo+gnWffE6S55alAZ3U45DbN9iBNk8IL/GqkWUEZrrefHHJT8mbUaYk7hYSB1VsJ176DsVpha9uAqOSpW0EiN5NNjrsdyM8FnGnkIEFI2DeyTzzk8gDXYYs9TBHE2r4Nsq8eJVzcuDShndaWa0gTcowlM+mDr6FNek/tQ19Pj/383dIMW4eVHAAlke3ZQsbM2tPF4gl7nxyO3yIKV8FSkavs2WHD3dndFl9c+TeF3jjD7wyLgl8scuZejjE584sVkWUcrR0u7VABBmn0jN9szq7bBO3St136jKX/9F0H0D/Z1wpE1IdvsfbCw+F/20QqMIyjR0pZwSDoz6WpB852TApI+jGEBH33ZOAMtQsYVTaphaILXfY1kBD8sl1sDn2e8t3MLW23cI1izbLe0fXw1qvaddvn3v8AUEsDBBQAAAAIALKGUV394GQsowcAAKoQAAAbAAAAc3JjL3BoeXNwcm9iZS96dW5hX3Rvb2xzLnB5jVdtb+M2Ev7uX8FTcVhpz9E66eWK88EfvJugm142CeykRZsLCFqibNWyKJCUnWyQ/37PkJKsZNNgBdiiqOFwXh4+MwqCYFaX7I+biyk7jA9ZXmZSyzKRLM21TGzxwEIjNpIloiiYMOxrXYoYb1RprK4Ty7M8G7JNXtaG2ZVkldQHWV5IluXLWksTxYPBqUhW0FzVlsUQhy6tc2lYbg1Tu5Kt8jSVJZMFNtQqpTclWaJu3y1Eat7dsdAZKIqdeDBsI8yaNlNGRkNmFFOlHDj7ElGyRG2lhkz5wMhMnebl0rBdbldwKXPeWa8iZnMJWVE454xbrWvamVUC04UshqSbfGI/X93EgyAIBplWG8Z5Vlu4xznLN5XSlomyVFbYHHEZDJo5ZdqRWdU2L7qnelFplUizf/9gvOJK2FWRL1qtV3gcDD6f/fz5ajqfswkbxceDwSCVGeMkasJoPGC4GnnKjnsWVQVpWh66jMFgJIXzKIZrFIEPLOhy/WH6+1HxoUCgavEBK42bEBZyfJHkgdOoJRwuSfHQaYcCKZdcbkURVw9BNwlkIPXmg79ztwmBJH4Qm8KLNTY0t8afQtRlsgodTDjAh9DXthvbTeUHy6oeI0MWiZcybYdWVsaN4fTR8ZAVauniM7lA/poQIXlzKygtSKnD0x7sTTrwyr1Fsv/TOOxRfaUqAHQlyrSQMaGgifGQtTEYMu8w9nZx5FopsqZNk1uRKc1Swlf4im+NmXS5vKVRvFljPvRhMpNrXUtseJ8by9XaPXq1aouNHrvVTQqCcWdSkNZ+D8w53e2WsVgYVdRWhtFwv76LC28O8fcvTYUVMf31l3Q5fbaoEd7+kzs64UZsacMMqwJQiwxequ0kldfWqt/H8i/0G7ncEJaNTLDmOB69onmjSiuWkjY3FnkWOuWHo9Fx0NNTG2cg7511ks9EYd6wtpGWdBTyouiteDUClZYNGmX6pnZQI7dqLREPsFghKkiP4BskwHO1gXk8yQgGhy9mQegVyMCdGrx291c2WOXLVSWM4auvkGpJCJro5NE63PyyJ/efbFLg8BZcFst7mdRWLApJh1OH7SmJ7tg/2G0WPK6fJo/bp8CdiPWQbelQqG2cW7nBYblz+mRJuFbQVm5zrco4UdVDc5IwdRt8ujmZ8l/P5mcfz0/5yemvZ59O58Ed1tCWIIqe6G/Ti5OP/MvlyakTQDAMWZf6k/wDu8Jpl2XKrGILyQQ7PGgpwSqdrKgs/KkWVGz88WbAgmGXNzPmeDfMQSsgCr3LDUqn1PlWeuag4tns4STJYao6Dx3nON6fn9/MvvBfLj/ysxNwJFmyU62I7Ihpg0qa456ooshTieLaeBjXFdImw/PLT9NzPpte/HcSjICw/ei3y9n5CZ+f/XE6CQ7x7CVfzHYgaK4v0/n16YxPT05meH/0UwyAxbS6eXF1ObueULSP/n08GiG3P7H3RMMu+lEv/le/X3++vLiaXn928UdSHS/KKv5T5WV4Szp6xBl52IBhMSL3ltKGfSVAYRDdRbGGWF6Fe317ol0Tpm7X+zHhCd3HOjZUBgy1BGHgAw9de/ZNZeGMXnsYgl7IYhSAsC0r2H0XRKSsnUHzgsTvC3sMOF7cnJ/3S2fvrasnIQ6M826CHzmcYqsJfm4stZ70VsyvTy5vrqOmWAKPvO1RwqVWNQ4w2ym99pT4dnEsObLTzRz6agn2fFEsvdoxAG8A7szdDQ1cD+fC7dsjJ9D2SGzWlM1H1wWWgP+YhUQpqJArTs8meupqaNO3bEp/SMhbQyRy16UR5Qv7GZ/AeoN+DTD3tvUqJiwCyzrJ8TMQg1ctmlO5z27bF1G4InQsWYBwPC6fgk7Ed2ux3lgtZYgc5ctSodlDSpQ2vcJLV5j6Xip4rVzvxciVjJz41kSAF9Bfh+ijW124ORuzKKaIOaithBEWh0Ih7AEtAAI96ry5jh3f0NHt6YIc42yB8MIQ8U19iIdtD9ZqaHQBke0QxT9wPdhkyf4ONroPPZiG7DD6hjz++iJ0TugPjIEiOwJpLBugTnw12ndwbl/Xw2IqaEmlwY3N8WmCrxJOA/fCjqjposeY/ppysVsRGPFNEKL5BHeGiKhhhHeXmPd8yCrX91NoovHe0h/YSoIsFlJYIn9gTC6UwudHqsp3FlaqNRH4V1l23vvNTSFlFf442ocdrDJhYd80dgBzCYL/GvWBTPwJ4ffsKEKQD+EQ+v7nkKk0CWUBY3Ec+2bWAAMFEUMJGmXhoyzG8Sh7ok8zsjwTOkLqsqI2qx400dxhUuQFtXnURT4NXxy+Dhz7AO1t0QnWVPFO5DaM+j40lEJRpnNJkX7ugYNmI/XdjW572XtirvAFNPBZSi2RvEf4/EkN0EsVIpHB8+Xhs80dBZAW7nmAW4Sj0bjT6Ei8Sux5e3AEsI7Gd89cRQz+9k2CSMfbJka3Bz86Zc/WaUENBD7JCSGn5AWy7PLb+0QBzzIY6vKGMxQiUpY96uQpGv+vfKStn4LvYh60nTAS5BvnypumxY5a3LA7+HRfPSx0nr4kFCBCy0KJtMkXmpoFPsknwelsdjl7EXKg67a/mJqAENtTYedUHahhp+euRvQrJ1YP/g9QSwMEFAAAAAgAl4ZRXQhrsgUIAgAAowcAABkAAABqb2JzL3Byb2JlMi9jb25maWdfQS5qc29uhVU9j8IwDN37K6rMDOSLltsQUlc6sJ0QqkrvrgJKRcsC4r9fkzpxmgscA3bs2H55dtJHFJOuqg7kI+bzpZwNy2PdHLph/RnFMblWxYlE8W6m95WX0UXnet1XrVoxHdbsv9ubdqrVueiOavFQSbqy6PvqqquorONfrMKGXzoKykCCmS5AJqNk4GccJPg5BQl+MQcJdrEcpYQ4KZXYzRAHm+CA7QzCGcBh4OZzryz4OfgFwJKQVQqQkHcB8QvmwwB8sJ9CfmqOu/Tqc68+0MRhnxAv8Bga0gmOCLCQtujLn2mfTCPYFBE1TPmEyxcVxbSyXPoMQCLD/MylJQk2SXqkeLPA0z+9Nsn9aRJeu+HQzEyfT79fiVsao/hpbsC+Kc7Vf9eAZGtBxixkza2GthyNOUX1btSsdVRm1FWGVpsgs/HbxGqp0TB7vrDaJiE+iQNiGkBMEQVFFBxVWyiTAWj2wBliy1Gz7hzxbqy6sRU3LIQXuVhbhhzNHjdLAhjvAYx4FuRvm78H7pAqfLjvLuAA2oY6c4G23BkWa3VxpW/RpAFcitsAj0gKNt6ZBodo3BoaEWcugsOAGbeh+WudWQsPfxKo5FyJldPKUMtDSO7YKbjjR/OdpPCJbPb9tS5O+jOonipSXtpavwDqfSFf9anq9m11Vd9KLcufW3Mc3Akzb8bo12lUluj5C1BLAwQUAAAACACXhlFd/XjVnQoCAACmBwAAGQAAAGpvYnMvcHJvYmUyL2NvbmZpZ19CLmpzb26FVUuPgjAQvvMrSM8e7Atwb8aEqxy8bQxhkV2NikTwovG/L4Vpp3Sr68F5tTNfv5mWRxCStqp25CPk80U0683jod61vf0ZhCG5VsUp//ohQbidDUvLyxil88HuqkZZTCqrzn+a2xBU1rloj8p4qDxtWXRddR0KqcTjX6i29b9kFJSBBDeNQMajZBBnHCTEOQUJcTEHCX6xGKWEfVIqsZ0hDjbBAcsZbGcAh0GYz52yEOcQFwBLQlYpQELeCPZHzIUB+GA9hfxUH3fh1OdOfaCJwzohXuDRNCQTHAFgIU3Rlftpn3Qj2BQR1Uy5hMsXFcW0sly4DEAizfzMpiX2Nkk6pDizwJM/vdbJ3WkSTrvh0ExPn0u/W4kbGoPwqW9AXhfn6r9rQNKVIGMWsuJGQ1+Gzoyietdq2lgq0+oyRa9JkJr9m9hoidYwexYZbR0Tl8QeMfUgpoiCIgqOqimUSg80c+AUsWWomXCGeNdGXZuKa+bDi1ysDEOWZo6bxh6Mdw9GPAvyt8neA7dIFS7cdxewB222WnOBvswaFuO1cSVv0SQeXIpbD49ICjbemgaLaFzqGxFrLrzDgBk3vvlrrFnzD3/sqWRdiaXVSl/LfUju2Cm440f9qaTwiazz7nooTsNnUD1VpLw0h+EFUO8L+T6cqjZvqqv6Vg6y3N/qYx+OmX4zxviQRmUJnr9QSwECFAMUAAAACACXhlFdBjTceu0JAADCGAAAGgAAAAAAAAAAAAAApIEAAAAAc3JjL3BoeXNwcm9iZS9ydW5fcHJvYmUucHlQSwECFAMUAAAACACyVFFdAAAAAAIAAAAAAAAAGQAAAAAAAAAAAAAApIElCgAAc3JjL3BoeXNwcm9iZS9fX2luaXRfXy5weVBLAQIUAxQAAAAIAHlVUV190ErxtwUAABENAAAWAAAAAAAAAAAAAACkgV4KAABzcmMvcGh5c3Byb2JlL3N5bnRoLnB5UEsBAhQDFAAAAAgAslRRXYG3sDDjBwAAMhEAACEAAAAAAAAAAAAAAKSBSRAAAHNyYy9waHlzcHJvYmUvbGVhZGZpZWxkX3NwaGVyZS5weVBLAQIUAxQAAAAIAJeGUV3HuESvrgkAAB4YAAAWAAAAAAAAAAAAAACkgWsYAABzcmMvcGh5c3Byb2JlL3Byb2JlLnB5UEsBAhQDFAAAAAgAsoZRXf3gZCyjBwAAqhAAABsAAAAAAAAAAAAAAKSBTSIAAHNyYy9waHlzcHJvYmUvenVuYV90b29scy5weVBLAQIUAxQAAAAIAJeGUV0Ia7IFCAIAAKMHAAAZAAAAAAAAAAAAAACkgSkqAABqb2JzL3Byb2JlMi9jb25maWdfQS5qc29uUEsBAhQDFAAAAAgAl4ZRXf141Z0KAgAApgcAABkAAAAAAAAAAAAAAKSBaCwAAGpvYnMvcHJvYmUyL2NvbmZpZ19CLmpzb25QSwUGAAAAAAgACAA9AgAAqS4AAAAA"
zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD))).extractall("/content/eegpp")
sys.path.insert(0, "/content/eegpp/src")
os.environ["PROBE_WORK"] = "/content/work"
print("code ready")

In [ ]:
# 4. Run part A then part B (each ~1 h). Finished parts are skipped on a re-run.
import json, time
from pathlib import Path
from physprobe.run_probe import run
for part in ("A", "B"):
    out = SAVE / f"part_{part}"
    if (out / "log.json").exists() and json.loads((out / "log.json").read_text()).get("ok"):
        print(f"part {part}: already done, skipping"); continue
    cfg = json.loads(Path(f"/content/eegpp/jobs/probe2/config_{part}.json").read_text())
    print(f"part {part}: starting ({cfg['n_trials']} recordings x {cfg['copies']} samples)", flush=True)
    t0 = time.time()
    log = run(cfg, out, mock=False)
    print(f"part {part}: {'done' if log.get('ok') else 'FAILED'} in {(time.time() - t0) / 60:.0f} min")
    if not log.get("ok"):
        print(log.get("error", "")[-2000:]); break

In [ ]:
# 5. Download the results (attach probe2_results.zip in the chat)
import shutil
zip_path = shutil.make_archive("/content/probe2_results", "zip", SAVE)
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("Download manually:", zip_path)